# Task 5: Train-Test Split

**The Goal**: Prevent the model from merely memorizing the data (overfitting).

**The Action**: Take the newly encoded, complex dataset and strictly divide it (80% for training, 20% for testing) before feeding it to any algorithm.

**The Research**: Train a new multivariable model on the 80% split and test it on the 20%. Document why evaluating a model on the exact same data it was trained on is a critical failure in data science.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

# 1. Load data and apply One-Hot Encoding (from Task 4)
df = pd.read_csv('ai4i2020.csv')
df_encoded = pd.get_dummies(df, columns=['Type'], drop_first=True)

# Select features (continuous + our newly encoded dummy variables)
features = ['Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]', 'Air temperature [K]', 'Type_L', 'Type_M']
X = df_encoded[features]
y = df_encoded['Process temperature [K]']

# 2. Train-Test Split (80% / 20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Total dataset size: {len(X)}")
print(f"Training set size: {len(X_train)} (80%)")
print(f"Testing set size: {len(X_test)} (20%)\n")

# 3. Train the multivariable model on the 80% split
model = LinearRegression()
model.fit(X_train, y_train)

# Predict on the Training Data
y_train_pred = model.predict(X_train)
train_mse = mean_squared_error(y_train, y_train_pred)

# Predict on the Testing Data (Unseen)
y_test_pred = model.predict(X_test)
test_mse = mean_squared_error(y_test, y_test_pred)

print(f"Training Mean Squared Error (MSE): {train_mse:.4f}")
print(f"Testing Mean Squared Error (MSE): {test_mse:.4f}")

### Research: Why Evaluating on Training Data is a Critical Failure

**The Data Science Cardinal Sin: "Overfitting" and "Memorization"**

The purpose of a machine learning model isn't just to plot points on a graph it has already seen; the goal is **generalization**—how accurately the model can predict outcomes for completely new, unseen data in the real world. 

If we evaluate the performance of a model exclusively on the exact same data it used to learn its internal mathematical weights:
1. **The open-book test analogy**: Evaluating on training data is like giving a student an exam where they already memorized the answers to the exact questions beforehand. They might score 100%, but it tells you nothing about their underlying understanding of the topic.
2. **False Confidence**: An overly complex model might draw a squiggly line that perfectly hits every single data point in the training set (a literal 0 error rate). As a data scientist, you might think you have the "perfect model." However, real-world data contains random noise. By strictly memorizing that noise, the model acts terribly when applied to new readings.

**The Solution:**
By completely withholding a piece of the dataset—the 20% `Testing Set`—we force the model to take a "closed-book test". Because it never had access to this new data during training, the testing MSE tells us how the model behaves in genuine, real-world deployment scenarios. If training error is very low but testing error is very high, we immediately know the model overfit the data and merely "memorized" it.